# 03 - Silver
Desaninha, tipa, trata nulos e deduplica os dados da Bronze. As queries estão em `sql/silver/`.

| Tabela | Grão (1 linha por...) | Técnica principal |
|---|---|---|
| `silver_estadios` | time | CSV de referência tipado |
| `silver_times` | time | `EXPLODE` |
| `silver_partidas` | partida | `EXPLODE` + datas UTC → Brasília + nulos + `ROW_NUMBER` |
| `silver_arbitros` | árbitro por partida | `LATERAL VIEW explode` duplo |
| `silver_clima_diario` | cidade por dia | `arrays_zip` + `EXPLODE` |

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import logging
import os
import sys

sys.path.insert(0, os.path.abspath(".."))
logging.basicConfig(level=logging.WARNING, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
logging.getLogger("src").setLevel(logging.INFO)

from src.config import load_settings
from src.transform.silver import build_silver

os.environ["FOOTBALL_DATA_API_KEY"] = dbutils.secrets.get(scope="fc", key="football_data_api_key")
settings = load_settings()

In [0]:
contagens = build_silver(spark, settings)
for tabela, linhas in contagens.items():
    print(f"{tabela}: {linhas} linha(s)")

In [0]:
%sql
SELECT
  COUNT(*)                           AS linhas,
  COUNT(DISTINCT partida_id)         AS partidas_unicas,
  COUNT_IF(encerrada)                AS encerradas,
  COUNT_IF(gols_mandante IS NULL)    AS sem_placar,
  COUNT_IF(resultado = 'A_DEFINIR')  AS a_definir
FROM workspace.fc.silver_partidas

In [0]:
%sql
SELECT rodada, data_hora_utc, data_hora_brasilia, data_partida,
       mandante, gols_mandante, gols_visitante, visitante, resultado
FROM workspace.fc.silver_partidas
ORDER BY data_hora_utc
LIMIT 10

In [0]:
%sql
SELECT cidade_id, COUNT(*) AS dias, MIN(data) AS primeiro_dia, MAX(data) AS ultimo_dia,
       ROUND(AVG(temp_media_c), 1) AS temp_media, ROUND(SUM(precipitacao_mm)) AS chuva_total_mm
FROM workspace.fc.silver_clima_diario
GROUP BY cidade_id
ORDER BY chuva_total_mm DESC

In [0]:
%sql
SELECT funcao, COUNT(*) AS registros, COUNT(DISTINCT arbitro_id) AS arbitros_distintos
FROM workspace.fc.silver_arbitros
GROUP BY funcao
ORDER BY registros DESC